# Lab 2: Word Embeddings and Cosine Similarity

**Course:** AAE-PT02M1 | **Monday, Lesson 5 of 10** | *Context Understanding via Embeddings*

---

## Recall from earlier

Last lesson, tokenisation gave each word an **integer ID**. But an ID carries **no meaning**: ID `1087` is not "closer" in meaning to `1088` than to any other number. This lesson fixes that by giving each token a **vector** that captures meaning.

## What this notebook does

This notebook takes the lab code from the lesson **exactly as it is written** and explains it step by step. The code cells are the lesson's code, split into small pieces so each piece can be explained. Nothing has been added to the code.

## The idea

Once words are turned into numbers, the system relies on **word embeddings**. An embedding assigns an array of numbers (a **vector**) to every word. Words with similar meanings sit **close to each other** in mathematical space. For example, `cough` and `fever` will have highly similar vector values because they frequently appear together in medical charts.

Consider the word `bank`. It can mean a financial institution or a river bank. The **surrounding words** help the model determine the correct meaning. This contextual understanding is what lets LLMs handle ambiguous or domain-specific language.

## Why this matters for AfyaPlus

Embeddings let the AI understand that **"I cannot breathe"** and **"my chest is tight"** are semantically similar even though they share no common words. The model groups dangerous symptom phrasings together, so guardrails trigger correctly however the patient words their distress.

> **Engineering reality check**
> In production, vectors are not just 3 numbers long. They use **1,536 or more dimensions**. These large dimensions are needed because medical language is complex, and a vector needs enough mathematical space to capture tiny differences between a *sharp pain* (emergency) and a *dull ache* (chronic). We use **3 dimensions** here so the maths is readable on the page.

## The goal of this lab

We simulate how a model uses **cosine similarity** to decide whether the word **`discharge`** refers to a patient leaving the hospital (administrative) or to an active symptom (clinical).

NumPy is already installed from the tokenisation lab, so no new setup is needed. Run the cells **in order**.

---

## Build it step by step

| Step | What happens |
|---|---|
| 1 | Represent words as vectors |
| 2 | Write the cosine-similarity function |
| 3 | Compare an ambiguous word |
| 4 | Read the result |

### Step 1: Represent words as vectors

Each word becomes a short list of numbers along meaningful axes. Similar words get similar numbers.

The lesson uses three axes, in this order:

| Position | Axis | A high value means |
|---|---|---|
| 0 | Clinical-Symptom | the word describes something medical |
| 1 | Administrative | the word is about paperwork, billing, hospital process |
| 2 | Time/Urgency | the word suggests something time-critical |

In a real model nobody labels the axes: the numbers are *learned* during training. The labelled axes here are a teaching aid.

In [ ]:
import numpy as np

# Simulated 3D word embeddings.
# Each dimension represents: [Clinical-Symptom, Administrative, Time/Urgency]
embeddings = {
    "fever":     np.array([0.9, 0.0, 0.3]),
    "pain":      np.array([0.8, 0.1, 0.4]),
    "discharge": np.array([0.5, 0.5, 0.1]), # Ambiguous word!
    "billing":   np.array([0.0, 0.9, 0.1]),
    "admit":     np.array([0.2, 0.8, 0.2])
}

### Explanation of Step 1

**`import numpy as np`** loads NumPy.

**`embeddings = { ... }`** creates a Python **dictionary**: a collection of key and value pairs. Here each **key** is a word (in quotes) and each **value** is a NumPy array of three numbers, which is that word's vector. You can look up a vector with, for example, `embeddings["fever"]`.

Read the vectors against the axes:

| Word | Clinical | Administrative | Urgency | What it tells us |
|---|---|---|---|---|
| fever | 0.9 | 0.0 | 0.3 | strongly clinical, not administrative |
| pain | 0.8 | 0.1 | 0.4 | strongly clinical, a bit more urgent |
| discharge | 0.5 | 0.5 | 0.1 | **ambiguous**: equal clinical and administrative pull |
| billing | 0.0 | 0.9 | 0.1 | strongly administrative |
| admit | 0.2 | 0.8 | 0.2 | mostly administrative |

`discharge` is deliberately ambiguous, as the comment in the code says. In hospitals it can mean a patient leaving (administrative) **or** fluid from a wound or eye (a clinical symptom). Running this cell prints nothing: it only creates the dictionary.

---

### Step 2: Write the cosine-similarity function

Cosine similarity measures the **angle between two vectors**. Values near **1.0** mean "very similar meaning", values near **0** mean "unrelated".

In [ ]:
def cosine_similarity(word1, word2):
    """Calculate how close two words are in embedding space."""
    vec1 = embeddings[word1]
    vec2 = embeddings[word2]
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

### Explanation of Step 2: how the calculation works

The function takes two words and returns one number. Here is each line.

**`vec1 = embeddings[word1]`** and **`vec2 = embeddings[word2]`**
Look up each word's vector in the dictionary.

**`np.dot(vec1, vec2)`** is the **dot product**. Multiply the two vectors position by position, then add up the results:

$$\mathbf{a}\cdot\mathbf{b} = a_1 b_1 + a_2 b_2 + a_3 b_3$$

The dot product is large when both vectors have big numbers in the *same* positions.

**`np.linalg.norm(vec)`** is the **norm**, which is the vector's length (Pythagoras in three dimensions): square every number, add them up, take the square root.

$$\|\mathbf{a}\| = \sqrt{a_1^2 + a_2^2 + a_3^2}$$

**The division.** The dot product alone has a flaw: longer vectors give bigger products even if they point the same way. Dividing by the two lengths **cancels out size** and leaves only the **angle** between the vectors:

$$\text{cosine similarity}(\mathbf{a},\mathbf{b}) = \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}$$

**How to read a score**

| Score | Meaning |
|---|---|
| near **1.0** | the vectors point the same way: very similar meaning |
| near **0** | the vectors are at right angles: unrelated |
| near **-1** | opposite directions (does not happen with the all-positive numbers used here) |

**Why not just use the dot product?** Because cosine looks at *direction* only. A vector ten times longer but pointing the same way gets the same cosine score, which is what we want when comparing meanings.

Running this cell prints nothing: it only defines the function.

---

### Step 3: Compare an ambiguous word

If `discharge` appears alongside `pain`, it is likely a clinical symptom. If it appears alongside `billing`, it is administrative. We score both pairings and let the numbers tell us.

In [ ]:
# If 'discharge' appears alongside 'pain', it is likely a clinical symptom.
# If it appears alongside 'billing', it is administrative.
print("--- Context-Disambiguation via Cosine Similarity ---")
print(f"Context Match (Discharge + Pain):    {cosine_similarity('discharge', 'pain'):.4f}")
print(f"Context Match (Discharge + Billing): {cosine_similarity('discharge', 'billing'):.4f}")

### Explanation of Step 3

Each `print` calls `cosine_similarity` with a pair of words and formats the answer with `:.4f`, which means "show 4 digits after the decimal point".

---

### Step 4: Read the result

You should see:

```
--- Context-Disambiguation via Cosine Similarity ---
Context Match (Discharge + Pain):    0.7624
Context Match (Discharge + Billing): 0.7113
```

**Checking the first score by hand** (`discharge` = [0.5, 0.5, 0.1], `pain` = [0.8, 0.1, 0.4]):

1. Dot product: 0.5 x 0.8 + 0.5 x 0.1 + 0.1 x 0.4 = 0.40 + 0.05 + 0.04 = **0.49**
2. Length of `discharge`: the square root of (0.25 + 0.25 + 0.01) = the square root of 0.51 = **0.7141**
3. Length of `pain`: the square root of (0.64 + 0.01 + 0.16) = the square root of 0.81 = **0.9**
4. Cosine: 0.49 / (0.7141 x 0.9) = 0.49 / 0.6427 = **0.7624**

**Checking the second score** (`billing` = [0.0, 0.9, 0.1]): the dot product is 0 + 0.45 + 0.01 = 0.46, the length of `billing` is the square root of 0.82 = 0.9055, so the cosine is 0.46 / (0.7141 x 0.9055) = **0.7113**.

**Interpretation:**

- `discharge` has roughly equal clinical and administrative pull.
- Paired with `pain` (highly clinical), the score skews clinical.
- Paired with `billing` (highly administrative), the score skews administrative.
- Here `pain` scores higher, so the clinical reading of `discharge` is the better match.

**Be honest about the margin.** The gap (0.7624 against 0.7113) is real but modest, because `discharge` was deliberately built to sit in the middle. That is the point of an ambiguous word: its scores are close, and the context decides. Real models do the same thing with thousands of dimensions and many context words rather than one.

This is the same disambiguation principle real LLMs use, just with thousands of dimensions instead of three.

---

## Try it yourself (by editing the lab code)

Edit the numbers in the Step 1 cell, re-run Step 1, then re-run Step 3:

1. Change the `discharge` vector to `[0.1, 0.9, 0.1]` (this is Self-check Question 2). Which score wins now?
2. Change `discharge` back to `[0.5, 0.5, 0.1]` and try `[0.9, 0.1, 0.1]`. Which score wins, and by how much?

Always restore the original vector afterwards so the rest of the notebook matches the explanations.

---

## Did you run the code? Output self-check

These three questions build in depth: first **recall** what your code printed, then **apply** the idea by predicting how a change affects the output, then **analyse** why it behaves that way. If you cannot answer the first from memory, run the code again before continuing.

**Question 1 (Level 1, Recall: what the code printed)**
The lab prints two cosine scores. Which pairing scores higher?

- A. Discharge with Billing
- B. Discharge with Pain
- C. The scores are equal
- D. Billing with Admit

**Question 2 (Level 2, Apply: predict the change)**
If you nudged `discharge` toward `[0.1, 0.9, 0.1]`, which score would now win?

- A. Discharge with Pain
- B. Discharge with Billing
- C. No change occurs
- D. Both drop to zero

**Question 3 (Level 3, Analyse: reason about why)**
Why can one guardrail catch "I cannot breathe" and "my chest is tight" together?

- A. They use the same token IDs
- B. They share exact keywords
- C. They have equal length
- D. Both map to nearby vectors

### Answers and explanations

**Question 1: B (Discharge with Pain).** 0.7624 against 0.7113. `pain` is strongly clinical and `discharge` has a clinical component, so they point more nearly the same way than `discharge` and `billing` do. (Option D describes a different pair, `billing` with `admit`, which is not what the lab printed.)

**Question 2: B (Discharge with Billing).** After the nudge, `discharge` is mostly administrative, so it points almost the same way as `billing`. When you run Step 3 with `[0.1, 0.9, 0.1]` you will see the `billing` score rise to about 0.99 and the `pain` score fall to about 0.26. The scores change smoothly. They do not drop to zero (D), and a changed vector certainly changes the result (C).

**Question 3: D (Both map to nearby vectors).** The two phrases share no words and no token IDs (A, B), and their length is irrelevant (C). A trained embedding model places both close together because they describe similar situations. A guardrail that watches a region of vector space then catches both, however the patient words the distress.

---

## Quick check: recall before you continue

**Two patients write "I cannot breathe" and "my chest is tight". They share no words. How can the model treat them as similar?**

<details>
<summary><b>Reveal answer</b></summary>

The model maps each phrase to a vector. Because both phrases appear in similar situations in the training data (emergencies, respiratory and cardiac distress), their vectors end up close together. The cosine similarity between them is high even though the words differ, so the model treats them as similar.

</details>

---

## Summary

| Step | What the code does | Key tool |
|---|---|---|
| 1 | Stores each word as a 3-number vector in a dictionary | `np.array`, dictionary |
| 2 | Defines cosine similarity: dot product divided by the two lengths | `np.dot`, `np.linalg.norm` |
| 3 | Scores `discharge` against `pain` and against `billing` | f-strings with `:.4f` |
| 4 | Reads the scores to see which meaning the context supports | comparing two numbers |

**Key ideas**

- An integer ID has no meaning. A **vector** does: similar meaning means nearby vectors.
- **Cosine similarity** compares direction, from about 0 (unrelated) to 1 (same).
- Context lets a model choose between the meanings of an ambiguous word.
- Production vectors have 1,536 or more dimensions to capture fine medical distinctions.

**Next:** open the companion notebook for the Challenge. When you finish, reply with **"Done"**.

*Coming up next: Prediction and Temperature. Watch the model turn meaning back into language, and meet the temperature dial.*